# ISO 19139 → JSON-LD Converter — LifeWatch ERIC v2.1
**Record type is auto-detected from the XML. No manual selection needed.**

| ISO 19139 record type | JSON-LD `@type` | Detected by |
|----------------------|-----------------|-------------|
| Workflow | `CreativeWork` | `hierarchyLevel=application` or "workflow" in title/keywords |
| VRE | `CreativeWork` | `hierarchyLevel=application` or "VRE"/"Virtual Lab" in title/keywords |
| Service → Action | `Action` | `hierarchyLevel=service` + registered I/O in `gmd:LW_Service` |
| Service → HowTo | `HowTo` | `hierarchyLevel=service` + "Marco-Bolo" keyword + no I/O |
| Service → CreativeWork | `CreativeWork` | `hierarchyLevel=service` + no Marco-Bolo + no I/O |

**Usage:** Runtime → Run all → upload your ISO 19139 XML → `output.jsonld` downloads automatically.

Override auto-detection: `result = run(xml_path='/content/file.xml', record_type='Action')`


In [ ]:
import subprocess, sys
for pkg in ["lxml", "tabulate"]:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("ISO19139_to_JsonLD v1 ready")


In [ ]:
import json, re, os
from lxml import etree
from tabulate import tabulate


In [ ]:
JSONLD_CONTEXT = {"@vocab": "https://schema.org/"}
CATALOGUE_HOST = "https://metadatacatalogue.lifewatch.eu"

# ISO 19139 namespace map
NS = {
    "gmd": "http://www.isotc211.org/2005/gmd",
    "gco": "http://www.isotc211.org/2005/gco",
    "srv": "http://www.isotc211.org/2005/srv",
    "gmx": "http://www.isotc211.org/2005/gmx",
    "gml": "http://www.opengis.net/gml/3.2",
    "xlink": "http://www.w3.org/1999/xlink",
}

# MD_ProgressCode -> schema.org creativeWorkStatus mapping
STATUS_MAP = {
    "completed":        ("Published", "http://purl.org/spar/pso/published"),
    "historicalArchive":("Archived",  "http://purl.org/spar/pso/archived"),
    "obsolete":         ("Archived",  "http://purl.org/spar/pso/archived"),
    "onGoing":          ("Published", "http://purl.org/spar/pso/published"),
    "underDevelopment": ("Draft",     "http://purl.org/spar/pso/draft"),
}

# CI_RoleCode values that map to "creator" in schema.org
CREATOR_ROLES = {"author", "owner", "originator", "principalInvestigator",
                 "resourceProvider", "processor", "publisher"}

UUID_RE = re.compile(
    r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}"
)


In [ ]:
def _xp(node, xpath, ns=NS):
    return node.xpath(xpath, namespaces=ns)

def _text(node, xpath, ns=NS):
    results = node.xpath(xpath, namespaces=ns)
    if results:
        t = (results[0].text or "").strip()
        return t if t else None
    return None

def _texts(node, xpath, ns=NS):
    return [(r.text or "").strip() for r in node.xpath(xpath, namespaces=ns)
            if (r.text or "").strip()]

def _attr(node, xpath, ns=NS):
    results = node.xpath(xpath, namespaces=ns)
    return results[0] if results else None


In [ ]:
def parse_contact(party_el):
    node = {}
    name = _text(party_el, "gmd:individualName/gco:CharacterString")
    org  = _text(party_el, "gmd:organisationName/gco:CharacterString")
    email= _text(party_el, ".//gmd:electronicMailAddress/gco:CharacterString")

    # Use empty string if org element exists but is empty (per sheet spec)
    org_el = party_el.xpath("gmd:organisationName/gco:CharacterString", namespaces=NS)
    org_raw = org_el[0].text if org_el else None  # None = element absent

    if name:
        parts = name.strip().split()
        node["@type"] = "Person"
        node["name"]  = name
        if len(parts) >= 2:
            node["givenName"]  = parts[0]
            node["familyName"] = " ".join(parts[1:])
        # Always include affiliation if org element exists (even if empty)
        if org_raw is not None:
            node["affiliation"] = {"@type": "Organization", "name": (org_raw or "").strip()}
    elif org:
        node["@type"] = "Organization"
        node["name"]  = org

    if email:
        node["email"] = email

    return node if node else None


def get_dates(root):
    dates = {"creation": None, "publication": None, "revision": None}
    for ci_date in _xp(root, ".//gmd:CI_Date"):
        code_val = _attr(ci_date, "gmd:dateType/gmd:CI_DateTypeCode/@codeListValue")
        date_val = _text(ci_date, "gmd:date/gco:Date")
        if not date_val:
            date_val = _text(ci_date, "gmd:date/gco:DateTime")
        if code_val in dates and date_val:
            dates[code_val] = date_val
    return dates


def get_distribution_urls(root):
    urls = []
    for url_el in _xp(root, ".//gmd:distributionInfo//gmd:CI_OnlineResource/gmd:linkage/gmd:URL"):
        u = (url_el.text or "").strip()
        if u and u not in urls:
            urls.append(u)
    return urls


def detect_record_type(root):
    """
    Auto-detect JSON-LD record type from ISO 19139 XML content.

    Decision tree (per spec):
      service + registered I/O (gmd:LW_Service or srv:SV_OperationMetadata) -> Action
      service + Marco-Bolo keyword + no I/O -> HowTo
      service + no Marco-Bolo + no I/O -> CreativeWork
      application / workflow / VRE signals -> CreativeWork
      default -> CreativeWork
    """
    hier = _attr(root, ".//gmd:hierarchyLevel/gmd:MD_ScopeCode/@codeListValue")
    title = (_text(root, ".//gmd:identificationInfo//gmd:title/gco:CharacterString") or "").lower()
    kws   = [k.lower() for k in _texts(root, ".//gmd:descriptiveKeywords//gmd:keyword/gco:CharacterString")]
    all_text = title + " " + " ".join(kws)

    # Registered I/O: standard srv: or LifeWatch ERIC custom gmd:LW_Service
    has_operations = (
        bool(_xp(root, ".//srv:SV_OperationMetadata")) or
        bool(_xp(root, ".//gmd:service/gmd:LW_Service/gmd:containOperations_service"))
    )
    is_marco_bolo = "marco-bolo" in all_text or "marco bolo" in all_text

    if hier == "service":
        if has_operations:
            return "Action",  "hierarchyLevel=service + registered I/O (LW_Service or srv:SV_OperationMetadata)"
        elif is_marco_bolo:
            return "HowTo",   "hierarchyLevel=service + Marco-Bolo keyword + no registered I/O"
        else:
            return "CreativeWork", "hierarchyLevel=service + not Marco-Bolo + no registered I/O"
    elif (hier == "application" or "workflow" in all_text or
          "vre" in all_text or "virtual lab" in all_text or "vlab" in all_text):
        return "CreativeWork", "hierarchyLevel=" + str(hier) + " / Workflow or VRE signals"
    else:
        return "CreativeWork", "Default (hierarchyLevel=" + str(hier) + ")"


In [ ]:
class ISO19139toJsonLD:
    """
    Convert ISO 19139 XML to JSON-LD per LifeWatch ERIC mapping spec.

    Supported @type values (choose based on record content):
      'Workflow'    -> CreativeWork  (Workflow records)
      'VRE'         -> CreativeWork  (Virtual Research Environment records)
      'Action'      -> Action        (Services with registered I/O in catalogue)
      'HowTo'       -> HowTo         (Marco-Bolo services without registered I/O)
      'CreativeWork'-> CreativeWork  (Other services without registered I/O)

    Mapping (all sheets):
      fileIdentifier    -> @id (/srv/api/records/{uuid}) + url (/catalog.search#/metadata/{uuid})
      title             -> name
      abstract          -> description
      creation date     -> dateCreated
      publication date  -> datePublished
      revision date     -> dateModified
      status            -> creativeWorkStatus (DefinedTerm with PSO URI)
      pointOfContact    -> creator[] (Person/Organization)
                        -> agent (for Action type)
      useLimitation     -> license
      keywords          -> keywords[] (plain strings)
      distribution URLs -> sameAs (plain string or array)
      Action type only  -> startTime + endTime from publication/creation date
    """

    def __init__(self, xml_source):
        if isinstance(xml_source, str) and os.path.exists(xml_source):
            with open(xml_source, 'rb') as fh: raw = fh.read()
        elif isinstance(xml_source, bytes): raw = xml_source
        else: raise TypeError('xml_source must be file path or XML bytes')
        parser = etree.XMLParser(remove_comments=True, recover=True)
        self._root = etree.fromstring(raw, parser=parser)
        self._log  = []
        self._result = None

    def _ok(self, field, target):
        self._log.append({'field': field, 'status': 'OK', 'target': target})

    def _partial(self, field, reason):
        self._log.append({'field': field, 'status': 'PARTIAL', 'target': reason})

    def _lost(self, field, reason):
        self._log.append({'field': field, 'status': 'LOST', 'target': reason})

    def _map_id(self, doc):
        uuid = _text(self._root, './/gmd:fileIdentifier/gco:CharacterString')
        if uuid:
            uuid = uuid.strip()
            doc['@id'] = CATALOGUE_HOST + '/srv/api/records/' + uuid
            doc['url'] = CATALOGUE_HOST + '/srv/eng/catalog.search#/metadata/' + uuid
            self._ok('fileIdentifier', '@id and url -> ' + uuid)
        else:
            self._lost('fileIdentifier', 'No fileIdentifier found')

    def _map_basic(self, doc):
        title = _text(self._root, './/gmd:identificationInfo//gmd:title/gco:CharacterString')
        if title:
            doc['name'] = title
            self._ok('title', 'name')
        else:
            self._lost('title', 'No title found')

        abstract = _text(self._root, './/gmd:identificationInfo//gmd:abstract/gco:CharacterString')
        if abstract:
            doc['description'] = abstract
            self._ok('abstract', 'description')
        else:
            self._lost('abstract', 'No abstract found')

    def _map_dates(self, doc, record_type):
        dates = get_dates(self._root)
        pub  = dates['publication']
        cre  = dates['creation']
        rev  = dates['revision']

        if record_type == 'Action':
            # Action: startTime + endTime (same value) from pub date, fallback to creation
            dt = pub or cre
            if dt:
                doc['startTime'] = dt
                doc['endTime']   = dt
                self._ok('publication/creation date', 'startTime + endTime = ' + dt)
            else:
                self._lost('date', 'No publication or creation date for startTime/endTime')
        else:
            if cre:
                doc['dateCreated'] = cre
                self._ok('creation date', 'dateCreated = ' + cre)
            if pub is not None:
                doc['datePublished'] = pub if pub else ''
                self._ok('publication date', 'datePublished = ' + (pub or '(empty)'))
            if rev is not None:
                doc['dateModified'] = rev if rev else ''
                self._ok('revision date', 'dateModified = ' + (rev or '(empty)'))

    def _map_status(self, doc):
        code = _attr(self._root,
            './/gmd:identificationInfo//gmd:status/gmd:MD_ProgressCode/@codeListValue')
        if code and code in STATUS_MAP:
            name, uri = STATUS_MAP[code]
            doc['creativeWorkStatus'] = {
                '@type': 'DefinedTerm',
                'name':  name,
                'url':   uri,
            }
            self._ok('status (' + code + ')', 'creativeWorkStatus -> ' + name)
        elif code:
            self._partial('status (' + code + ')',
                          'Unknown status code - not mapped')
        else:
            self._lost('status', 'No MD_ProgressCode found')

    def _map_contacts(self, doc, record_type):
        parties = _xp(self._root, './/gmd:pointOfContact/gmd:CI_ResponsibleParty')
        if not parties:
            self._lost('pointOfContact', 'No CI_ResponsibleParty elements')
            return

        creators = []
        for party in parties:
            node = parse_contact(party)
            if node:
                creators.append(node)

        if not creators:
            self._lost('pointOfContact', 'No parseable contact found')
            return

        if record_type == 'Action':
            # Action type uses 'agent' instead of 'creator'
            doc['agent'] = creators[0] if len(creators) == 1 else creators
            self._ok('pointOfContact', 'agent (Action type)')
        else:
            doc['creator'] = creators[0] if len(creators) == 1 else creators
            self._ok('pointOfContact', 'creator (Person/Organization)')

        # provider is always the hardcoded LifeWatch ERIC organisation block (all record types)
        doc['provider'] = {
            "@type": "Organization",
            "name": "e-Science and Technology European Infrastructure for Biodiversity and Ecosystem Research",
            "alternateName": "LifeWatch ERIC",
            "identifier": "https://ror.org/04c04g438",
            "url": "https://www.lifewatch.eu",
            "email": "communications@lifewatch.eu"
        }
        self._ok('provider', 'provider (hardcoded LifeWatch ERIC organisation block)')

    def _map_license(self, doc, record_type):
        if record_type == 'Action':
            # License not mapped for Action type per spec
            self._partial('license', 'Not mapped for Action type per spec')
            return
        lic = _text(self._root,
            './/gmd:resourceConstraints//gmd:useLimitation/gco:CharacterString')
        if lic:
            doc['license'] = lic
            self._ok('useLimitation', 'license = ' + lic)
        else:
            self._lost('license', 'No useLimitation found')

    def _map_keywords(self, doc, record_type):
        if record_type == 'Action':
            # Keywords not mapped for Action type per spec
            self._partial('keywords', 'Not mapped for Action type per spec')
            return
        kws = _texts(self._root,
            './/gmd:descriptiveKeywords//gmd:keyword/gco:CharacterString')
        if kws:
            doc['keywords'] = kws
            self._ok('keyword', 'keywords[] = ' + str(kws))
        else:
            self._lost('keyword', 'No keywords found')

    def _map_distribution(self, doc, record_type='CreativeWork'):
        """
        Per spec:
          Action: sameAs = last URL only (the runnable workflow/NaaVRE URL, not source code)
          All others: sameAs = all URLs (string if 1, array if multiple)
        """
        urls = get_distribution_urls(self._root)
        if not urls:
            self._lost('distributionInfo/URL', 'No online URLs found')
            return
        if record_type == 'Action' and len(urls) > 1:
            doc['sameAs'] = urls[-1]
            self._ok('distributionInfo/CI_OnlineResource/URL',
                     'sameAs = ' + urls[-1] + ' (last URL - workflow link)')
        else:
            doc['sameAs'] = urls[0] if len(urls) == 1 else urls
            self._ok('distributionInfo/CI_OnlineResource/URL',
                     'sameAs = ' + str(doc['sameAs']))

    def convert(self, record_type=None):
        """
        Convert ISO 19139 XML to JSON-LD.
        Auto-detects record type if not provided.
        Override: record_type='Action' | 'HowTo' | 'Workflow' | 'VRE' | 'CreativeWork'
        """
        self._log = []

        # Auto-detect if not provided
        if record_type is None:
            record_type, reason = detect_record_type(self._root)
            print('Auto-detected: ' + record_type + ' (' + reason + ')')
        else:
            print('Using record type: ' + record_type)

        type_map = {
            'Workflow':     'CreativeWork',
            'VRE':          'CreativeWork',
            'Action':       'Action',
            'HowTo':        'HowTo',
            'CreativeWork': 'CreativeWork',
        }
        schema_type = type_map.get(record_type, 'CreativeWork')

        doc = {
            '@context': JSONLD_CONTEXT,
            '@type':    schema_type,
        }

        self._map_id(doc)
        self._map_basic(doc)
        self._map_dates(doc, record_type)
        self._map_status(doc)
        self._map_contacts(doc, record_type)
        self._map_license(doc, record_type)
        self._map_keywords(doc, record_type)
        self._map_distribution(doc, record_type)

        self._result = doc
        return doc

    def save(self, path):
        if self._result is None: self.convert()
        with open(path, 'w', encoding='utf-8') as fh:
            json.dump(self._result, fh, indent=2, ensure_ascii=False)
        print('Saved -> ' + path)

    def print_loss_report(self):
        if not self._log: self.convert()
        mapped  = sum(1 for r in self._log if r['status'] == 'OK')
        partial = sum(1 for r in self._log if r['status'] == 'PARTIAL')
        lost    = sum(1 for r in self._log if r['status'] == 'LOST')
        total   = len(self._log)
        pct     = round(100 * mapped / total) if total else 0
        sep = '=' * 80
        print(sep)
        print('MAPPING REPORT  ISO 19139 -> JSON-LD  (LifeWatch ERIC spec v2.1)')
        print(sep)
        print('Fields evaluated : ' + str(total))
        print('Fully mapped     : ' + str(mapped) + '  (' + str(pct) + '%)')
        print('Partial          : ' + str(partial))
        print('Lost             : ' + str(lost))
        print(sep)
        rows = [[r['field'], r['status'], r['target']] for r in self._log]
        print(tabulate(rows,
                       headers=['ISO 19139 Field', 'Status', 'Mapped to / Reason'],
                       tablefmt='simple'))
        print(sep)


print('ISO19139toJsonLD v2.1 loaded - auto-detects record type (Workflow/VRE/Action/HowTo/CreativeWork)')
print('  provider <- hardcoded LifeWatch ERIC block (all record types)')
print('  affiliation always included per sheet spec')


In [ ]:
def run(xml_path=None, record_type=None, output_path="output.jsonld"):
    """
    Main entry point. Record type is AUTO-DETECTED from the XML.
    Override: record_type='Action'|'HowTo'|'Workflow'|'VRE'|'CreativeWork'
    """
    # Step 1: Get XML
    if xml_path is None:
        try:
            from google.colab import files
            print("Upload your ISO 19139 XML file ...")
            uploaded = files.upload()
            if not uploaded:
                print("No file uploaded.")
                return None
            fname     = list(uploaded.keys())[0]
            xml_bytes = uploaded[fname]
            print("Uploaded: " + fname)
        except Exception:
            print("Not in Colab. Use: result = run(xml_path='/content/your_file.xml', record_type='Workflow')")
            return None
    else:
        fname = os.path.basename(xml_path)
        with open(xml_path, "rb") as fh:
            xml_bytes = fh.read()
        print("Loaded: " + fname)

    # Step 2: Convert (auto-detects type from XML; override with record_type param)
    print("Converting ...")
    conv   = ISO19139toJsonLD(xml_bytes)
    result = conv.convert(record_type=record_type)
    conv.save(output_path)

    # Step 4: Preview
    try:
        from IPython.display import display, JSON
        print("JSON-LD preview:")
        display(JSON(result))
    except Exception:
        print(json.dumps(result, indent=2, ensure_ascii=False)[:3000])

    # Step 5: Loss report
    conv.print_loss_report()

    # Step 6: Download
    try:
        from google.colab import files
        files.download(output_path)
        print("Downloaded: " + output_path)
    except Exception:
        print("Output saved to: " + output_path)

    return result


## Run
Click **Runtime → Run all**, upload your ISO 19139 XML — **record type is detected automatically**.

To override detection:
```python
result = run(xml_path='/content/your_file.xml', record_type='Action')
# Options: 'Workflow', 'VRE', 'Action', 'HowTo', 'CreativeWork'
```


In [ ]:
result = run()
